# Using the BaseCallable Module in baseobjects

## Introduction

The `BaseCallable` module provides an abstract class and its derivatives for creating customizable callable objects in Python. These classes implement the necessary magic methods and provide utilities for wrapping functions, binding methods to instances, and handling coroutines.

This module is particularly useful for creating decorators, method factories, and other advanced programming patterns that require customizable callable objects with specific behaviors. It handles edge cases like proper pickling, coroutine support, and attribute preservation that are often overlooked in custom callable implementations.

This tutorial will guide you through:
- Understanding the purpose and design of `BaseCallable`
- Creating custom callable objects
- Wrapping existing functions
- Handling coroutines
- Implementing the descriptor protocol for method binding

**Prerequisites:**
- Basic understanding of Python's callable objects and functions
- Familiarity with Python's descriptor protocol
- Knowledge of the `BaseReducible` class from the baseobjects package

### Table of Contents

- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Module Interaction](#Module-Interaction)
- [Advanced Features](#Advanced-Features)
- [Examples](#Examples)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting-/-FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)


## Importing the Module


In [11]:
from baseobjects.bases.basecallable import BaseCallable
import inspect


## Core Functionality

The `BaseCallable` class is an abstract class that implements the core functionality for creating callable objects in Python. It wraps an existing function or callable and implements the necessary protocols to make the wrapper behave like the wrapped function, including attribute copying, docstring preservation, and proper handling of coroutines.

### Key Features

1. **Function Wrapping**: Wraps existing functions and preserves their attributes
2. **Coroutine Support**: Properly handles coroutines, allowing async functions to be wrapped
3. **Descriptor Protocol**: Implements the descriptor protocol for method binding
4. **Attribute Preservation**: Preserves function attributes, including docstrings and annotations

Let's create a simple callable object using `BaseCallable`:


In [12]:
# Define a function to wrap
def greet(name):
    """A simple greeting function."""
    return f"Hello, {name}!"

# Create a BaseCallable that wraps the greet function
callable_greet = BaseCallable(greet)

# Call the callable object
print(callable_greet("World"))

# Check if the docstring is preserved
print(f"Docstring: {callable_greet.__doc__}")


Hello, World!
Docstring: A simple greeting function.


### Wrapping Functions with Attributes

One of the key features of `BaseCallable` is its ability to preserve function attributes. Let's create a function with custom attributes and see how `BaseCallable` preserves them:


In [13]:
# Define a function with custom attributes
def calculate(x, y, operation="add"):
    """Perform a calculation on two numbers."""
    if operation == "add":
        return x + y
    elif operation == "subtract":
        return x - y
    elif operation == "multiply":
        return x * y
    elif operation == "divide":
        return x / y
    else:
        raise ValueError(f"Unknown operation: {operation}")

# Add custom attributes to the function
calculate.supported_operations = ["add", "subtract", "multiply", "divide"]
calculate.author = "BaseObjects Team"

# Create a BaseCallable that wraps the calculate function
callable_calc = BaseCallable(calculate)

# Call the callable object
print(f"5 + 3 = {callable_calc(5, 3)}")
print(f"5 - 3 = {callable_calc(5, 3, 'subtract')}")
print(f"5 * 3 = {callable_calc(5, 3, 'multiply')}")
print(f"5 / 3 = {callable_calc(5, 3, 'divide')}")

# Check if the custom attributes are preserved
print(f"Supported operations: {callable_calc.supported_operations}")
print(f"Author: {callable_calc.author}")


5 + 3 = 8
5 - 3 = 2
5 * 3 = 15
5 / 3 = 1.6666666666666667
Supported operations: ['add', 'subtract', 'multiply', 'divide']
Author: BaseObjects Team


### Handling Coroutines

`BaseCallable` properly handles coroutines, allowing async functions to be wrapped without losing their async behavior. Let's create an async function and wrap it with `BaseCallable`:


In [14]:
import asyncio

# Import nest_asyncio to allow coroutines in Jupyter Notebooks
import nest_asyncio
nest_asyncio.apply()

# Define an async function
async def fetch_data(url):
    """Simulate fetching data from a URL."""
    print(f"Fetching data from {url}...")
    await asyncio.sleep(1)  # Simulate network delay
    return f"Data from {url}"

# Create a BaseCallable that wraps the async function
callable_fetch = BaseCallable(fetch_data)

# Check if the wrapped function is a coroutine
print(f"Is coroutine: {callable_fetch.is_coroutine}")

# Define a function to run the async function
async def run_async():
    result = await callable_fetch("https://example.com")
    print(f"Result: {result}")

# Run the async function
asyncio.run(run_async())



Is coroutine: True
Fetching data from https://example.com...
Result: Data from https://example.com


### Method Binding

`BaseCallable` implements the descriptor protocol, which allows it to be bound to instances when accessed as attributes. This is useful for creating method-like objects that can be bound to class instances:


In [15]:
# Define a class with a method
class Person:
    def __init__(self, name):
        self.name = name
    
    def greet(self, other):
        return f"{self.name} says hello to {other}!"

# Create a BaseCallable that wraps the greet method
callable_greet = BaseCallable(Person.greet)

# Create a Person instance
alice = Person("Alice")

# Bind the callable to the instance
bound_greet = callable_greet.__get__(alice, Person)

# Call the bound method
print(bound_greet("Bob"))


Alice says hello to Bob!


### Converting to a Function

`BaseCallable` provides a method to convert the callable object to a standard Python function. This is useful when you need to pass the callable to a function that expects a regular function:


In [16]:
# Create a BaseCallable
def add(x, y):
    """Add two numbers."""
    return x + y

callable_add = BaseCallable(add)

# Convert to a function
func_add = callable_add.as_function()

# Check the type of the function
print(f"Type of func_add: {type(func_add)}")

# Call the function
print(f"5 + 3 = {func_add(5, 3)}")

# Check if the docstring is preserved
print(f"Docstring: {func_add.__doc__}")


Type of func_add: <class 'function'>
5 + 3 = 8
Docstring: Add two numbers.


## Module Interaction

The `BaseCallable` module interacts with other modules in the baseobjects package, particularly `BaseReducible`. These interactions provide enhanced functionality for callable objects.

### Interaction with BaseReducible

`BaseCallable` inherits from `BaseReducible`, which means it also inherits all the functionality of `BaseReducible`, including pickling and unpickling:


In [17]:
import pickle

# Define a simple function
def square(x):
    """Return the square of a number."""
    return x * x

# Create a BaseCallable that wraps the square function
callable_square = BaseCallable(square)

# Pickle the callable object
pickled_callable = pickle.dumps(callable_square)
print(f"Pickled data (bytes): {pickled_callable[:30]}... (truncated)")

# Unpickle the callable object
unpickled_callable = pickle.loads(pickled_callable)

# Call the unpickled callable
print(f"5² = {unpickled_callable(5)}")


Pickled data (bytes): b'\x80\x04\x95\xef\x00\x00\x00\x00\x00\x00\x00\x8c\x1ebaseobjects.bases'... (truncated)
5² = 25


## Advanced Features

The `BaseCallable` class provides several advanced features that make it powerful for creating custom callable objects.

### Creating a Custom Callable Class

You can create your own custom callable class by inheriting from `BaseCallable` and overriding its methods:


In [ ]:
class LoggingCallable(BaseCallable):
    """A callable that logs its calls."""
    
    def __init__(self, func=None, log_prefix="CALL", *args, **kwargs):
        super().__init__(func, *args, **kwargs)
        self.log_prefix = log_prefix
        self.call_count = 0
    
    def __call__(self, *args, **kwargs):
        self.call_count += 1
        print(f"{self.log_prefix} #{self.call_count}: {self.__wrapped__.__name__}({args}, {kwargs})")
        result = self.__wrapped__(*args, **kwargs)
        print(f"{self.log_prefix} #{self.call_count} result: {result}")
        return result

# Create a LoggingCallable that wraps a function
def multiply(x, y):
    """Multiply two numbers."""
    return x * y

logging_multiply = LoggingCallable(multiply, log_prefix="MULTIPLY")

# Call the logging callable
result1 = logging_multiply(5, 3)
result2 = logging_multiply(7, 2)

print(f"Total calls: {logging_multiply.call_count}")


### Handling Coroutines in Custom Callables

When creating custom callable classes, you need to handle coroutines properly. `BaseCallable` provides the `is_coroutine` property to check if the wrapped function is a coroutine:


In [ ]:
class AsyncLoggingCallable(BaseCallable):
    """A callable that logs its calls and handles coroutines."""
    
    def __init__(self, func=None, log_prefix="ASYNC_CALL", *args, **kwargs):
        super().__init__(func, *args, **kwargs)
        self.log_prefix = log_prefix
        self.call_count = 0
    
    async def __call__(self, *args, **kwargs):
        self.call_count += 1
        print(f"{self.log_prefix} #{self.call_count}: {self.__wrapped__.__name__}({args}, {kwargs})")
        
        if self.is_coroutine:
            result = await self.__wrapped__(*args, **kwargs)
        else:
            result = self.__wrapped__(*args, **kwargs)
            
        print(f"{self.log_prefix} #{self.call_count} result: {result}")
        return result

# Create an AsyncLoggingCallable that wraps an async function
async def fetch_user(user_id):
    """Simulate fetching a user from a database."""
    await asyncio.sleep(0.5)  # Simulate database query
    return {"id": user_id, "name": f"User {user_id}"}

async_logging_fetch = AsyncLoggingCallable(fetch_user, log_prefix="FETCH_USER")

# Define a function to run the async function
async def run_async_logging():
    user1 = await async_logging_fetch(123)
    user2 = await async_logging_fetch(456)
    print(f"Total calls: {async_logging_fetch.call_count}")

# Run the async function
asyncio.run(run_async_logging())


## Examples

Let's explore some practical examples of using the `BaseCallable` module.

### Creating a Memoization Decorator

We can use `BaseCallable` to create a memoization decorator that caches function results:


In [21]:
class Memoize(BaseCallable):
    """A decorator that caches function results."""
    
    def __init__(self, func=None, *args, **kwargs):
        super().__init__(func, *args, **kwargs)
        self.cache = {}
    
    def __call__(self, *args, **kwargs):
        # Create a key from the arguments
        key = str(args) + str(sorted(kwargs.items()))
        
        # Check if the result is already in the cache
        if key not in self.cache:
            self.cache[key] = self.__wrapped__(*args, **kwargs)
            print(f"Cache miss for {self.__wrapped__.__name__}{args}")
        else:
            print(f"Cache hit for {self.__wrapped__.__name__}{args}")
            
        return self.cache[key]

# Create a memoization decorator
def memoize(func):
    return Memoize(func)

# Use the decorator on a recursive function
@memoize
def fibonacci(n):
    """Calculate the nth Fibonacci number."""
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)

# Calculate some Fibonacci numbers
print(f"fibonacci(10) = {fibonacci(10)}")
print(f"fibonacci(10) = {fibonacci(10)}")
print(f"fibonacci(11) = {fibonacci(11)}")


Cache miss for fibonacci(1,)
Cache miss for fibonacci(0,)
Cache miss for fibonacci(2,)
Cache hit for fibonacci(1,)
Cache miss for fibonacci(3,)
Cache hit for fibonacci(2,)
Cache miss for fibonacci(4,)
Cache hit for fibonacci(3,)
Cache miss for fibonacci(5,)
Cache hit for fibonacci(4,)
Cache miss for fibonacci(6,)
Cache hit for fibonacci(5,)
Cache miss for fibonacci(7,)
Cache hit for fibonacci(6,)
Cache miss for fibonacci(8,)
Cache hit for fibonacci(7,)
Cache miss for fibonacci(9,)
Cache hit for fibonacci(8,)
Cache miss for fibonacci(10,)
fibonacci(10) = 55
Cache hit for fibonacci(10,)
fibonacci(10) = 55
Cache hit for fibonacci(10,)
Cache hit for fibonacci(9,)
Cache miss for fibonacci(11,)
fibonacci(11) = 89


### Creating a Retry Decorator

We can use `BaseCallable` to create a retry decorator that retries a function if it raises an exception:


In [22]:
import random

class Retry(BaseCallable):
    """A decorator that retries a function if it raises an exception."""
    
    def __init__(self, func=None, max_retries=3, exceptions=(Exception,), *args, **kwargs):
        super().__init__(func, *args, **kwargs)
        self.max_retries = max_retries
        self.exceptions = exceptions
    
    def __call__(self, *args, **kwargs):
        retries = 0
        while retries <= self.max_retries:
            try:
                return self.__wrapped__(*args, **kwargs)
            except self.exceptions as e:
                retries += 1
                if retries > self.max_retries:
                    raise
                print(f"Retry {retries}/{self.max_retries} for {self.__wrapped__.__name__} due to {type(e).__name__}: {e}")

# Create a retry decorator
def retry(max_retries=3, exceptions=(Exception,)):
    def decorator(func):
        return Retry(func, max_retries=max_retries, exceptions=exceptions)
    return decorator

# Use the decorator
@retry(max_retries=5, exceptions=(ValueError,))
def unstable_function():
    """A function that sometimes raises an exception."""
    if random.random() < 0.7:
        raise ValueError("Random failure")
    return "Success!"

# Call the decorated function
try:
    result = unstable_function()
    print(f"Result: {result}")
except ValueError as e:
    print(f"Function failed after all retries: {e}")


Retry 1/5 for unstable_function due to ValueError: Random failure
Retry 2/5 for unstable_function due to ValueError: Random failure
Retry 3/5 for unstable_function due to ValueError: Random failure
Retry 4/5 for unstable_function due to ValueError: Random failure
Retry 5/5 for unstable_function due to ValueError: Random failure
Result: Success!


### Creating a Parameter Validation Decorator

We can use `BaseCallable` to create a decorator that validates function parameters:


In [23]:
class ValidateParams(BaseCallable):
    """A decorator that validates function parameters."""
    
    def __init__(self, func=None, validators=None, *args, **kwargs):
        super().__init__(func, *args, **kwargs)
        self.validators = validators or {}
    
    def __call__(self, *args, **kwargs):
        # Get the parameter names
        sig = inspect.signature(self.__wrapped__)
        bound_args = sig.bind(*args, **kwargs)
        bound_args.apply_defaults()
        
        # Validate each parameter
        for param_name, param_value in bound_args.arguments.items():
            if param_name in self.validators:
                validator = self.validators[param_name]
                if not validator(param_value):
                    raise ValueError(f"Invalid value for parameter '{param_name}': {param_value}")
        
        return self.__wrapped__(*args, **kwargs)

# Create a validate_params decorator
def validate_params(validators):
    def decorator(func):
        return ValidateParams(func, validators=validators)
    return decorator

# Use the decorator
@validate_params({
    'age': lambda x: isinstance(x, int) and x >= 0,
    'name': lambda x: isinstance(x, str) and len(x) > 0
})
def register_user(name, age):
    """Register a user with the given name and age."""
    return f"User {name} (age {age}) registered successfully"

# Call the decorated function with valid parameters
try:
    result = register_user("Alice", 30)
    print(result)
except ValueError as e:
    print(f"Validation error: {e}")

# Call the decorated function with invalid parameters
try:
    result = register_user("", -5)
    print(result)
except ValueError as e:
    print(f"Validation error: {e}")


User Alice (age 30) registered successfully
Validation error: Invalid value for parameter 'name': 


## API Highlights

Here are the key components of the `BaseCallable` module API:

### BaseCallable
- `__init__(func=None, *args, init=True, **kwargs)`: Initialize a new BaseCallable instance
- `__func__`: Property to get/set the wrapped function
- `is_coroutine`: Property to check if the wrapped function is a coroutine
- `bind_builtin(instance=None, owner=None)`: Creates a method bound to an instance using the builtin method
- `bind_wrapped(instance=None, owner=None)`: Creates a method of the wrapped function bound to an instance
- `call_wrapped(*args, **kwargs)`: Calls the wrapped function with the provided arguments
- `as_function()`: Creates a standard Python function that wraps this callable object

For more detailed information, consult the full API documentation.


## Troubleshooting / FAQs

### Q: Why use BaseCallable instead of just creating a callable class?

A: `BaseCallable` provides several advantages over creating a custom callable class from scratch:
1. It preserves function attributes, including docstrings and annotations
2. It properly handles coroutines
3. It implements the descriptor protocol for method binding
4. It provides utilities for binding to instances and converting to functions
5. It inherits from `BaseReducible`, which provides proper pickling support

### Q: How does BaseCallable handle coroutines?

A: `BaseCallable` detects if the wrapped function is a coroutine using `iscoroutinefunction` and sets the `_is_coroutine` flag accordingly. When converting to a function using `as_function()`, it creates an async function if the wrapped function is a coroutine.

### Q: Can I use BaseCallable with built-in functions?

A: Yes, `BaseCallable` can wrap any callable object, including built-in functions. However, some built-in functions may not have all the attributes that `BaseCallable` tries to copy, which could result in AttributeError exceptions during initialization. In such cases, you may need to handle these exceptions or use a try-except block.


## Conclusion and Next Steps

In this tutorial, we've explored the `BaseCallable` module and its primary class, `BaseCallable`. We've seen how this class provides a foundation for creating custom callable objects in Python, with features like function wrapping, coroutine support, method binding, and attribute preservation.

The `BaseCallable` class is particularly useful for creating decorators, method factories, and other advanced programming patterns that require customizable callable objects with specific behaviors. It handles edge cases like proper pickling, coroutine support, and attribute preservation that are often overlooked in custom callable implementations.

### Next Steps

- Explore the `BaseMethod` and `BaseFunction` modules, which extend `BaseCallable` to create method-like and function-like objects
- Check out the examples in the baseobjects package that demonstrate more advanced uses of `BaseCallable`
- Try creating your own custom callable classes by extending `BaseCallable`
- Consult the full API documentation for more detailed information on the `BaseCallable` module